# H3 Clustering for iSamples Geospatial Data

## What is H3?

[H3](https://h3geo.org/) is Uber's **hierarchical hexagonal indexing system** that partitions the
entire Earth's surface into hexagonal cells at 16 resolutions (0-15). Key properties:

- **Hierarchical**: Each parent cell contains ~7 children at the next resolution
- **Hexagonal**: Hexagons tile the plane with equal-area cells and uniform neighbor distances
- **Multi-resolution**: Zoom from continent-scale (res 0, ~4.4M km2) to sub-meter (res 15)

| Resolution | Hex Edge Length | Approx. Area | Use Case |
|-----------|----------------|-------------|----------|
| 4 | ~22 km | ~1,770 km2 | Country/region overview |
| 6 | ~3.2 km | ~36 km2 | City-level clustering |
| 8 | ~460 m | ~0.74 km2 | Neighborhood detail |

The iSamples wide parquet with H3 columns (`h3_res4`, `h3_res6`, `h3_res8`) covers
~11.96M of 20.7M rows — those with valid latitude/longitude coordinates.

This notebook demonstrates:
1. H3 cell statistics at multiple resolutions
2. Cluster visualization colored by dominant source
3. Multi-resolution comparison
4. Performance gains of clustering vs raw points
5. Hierarchical drill-down from coarse to fine cells

In [ ]:
import duckdb
import time
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Data URL
WIDE_H3_URL = "https://pub-a18234d962364c22a50c787b7ca09fa5.r2.dev/isamples_202601_wide_h3.parquet"

# Initialize DuckDB with H3 extension
con = duckdb.connect()
con.execute("INSTALL h3 FROM community; LOAD h3;")

# Source colors (RGBA)
SOURCE_COLORS = {
    'SESAR': [51, 102, 204, 200],
    'OPENCONTEXT': [220, 57, 18, 200],
    'GEOME': [16, 150, 24, 200],
    'SMITHSONIAN': [255, 153, 0, 200],
}
DEFAULT_COLOR = [128, 128, 128, 200]

print("Setup complete. DuckDB H3 extension loaded.")

## H3 Cell Statistics

How many distinct hexagonal cells exist at each resolution, and how does
the point distribution look?

In [ ]:
# Overall H3 coverage stats
stats = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(h3_res4) AS rows_with_h3,
        ROUND(100.0 * COUNT(h3_res4) / COUNT(*), 1) AS pct_with_h3,
        COUNT(DISTINCT h3_res4) AS distinct_res4,
        COUNT(DISTINCT h3_res6) AS distinct_res6,
        COUNT(DISTINCT h3_res8) AS distinct_res8
    FROM read_parquet('{WIDE_H3_URL}')
""").df()

print("iSamples H3 Coverage")
print("=" * 50)
print(f"Total rows:          {stats['total_rows'].iloc[0]:>12,}")
print(f"Rows with H3:        {stats['rows_with_h3'].iloc[0]:>12,} ({stats['pct_with_h3'].iloc[0]}%)")
print(f"Distinct res4 cells: {stats['distinct_res4'].iloc[0]:>12,}")
print(f"Distinct res6 cells: {stats['distinct_res6'].iloc[0]:>12,}")
print(f"Distinct res8 cells: {stats['distinct_res8'].iloc[0]:>12,}")

# Res6 cluster aggregation (~3.2 km hexagons)
print("\nComputing res6 clusters...")
t0 = time.time()
clusters = con.sql(f"""
    SELECT
        h3_res6,
        COUNT(*) AS sample_count,
        AVG(latitude) AS lat,
        AVG(longitude) AS lng,
        MODE(n) AS dominant_source,
        COUNT(DISTINCT n) AS source_count
    FROM read_parquet('{WIDE_H3_URL}')
    WHERE h3_res6 IS NOT NULL
    GROUP BY h3_res6
    ORDER BY sample_count DESC
""").df()
cluster_ms = (time.time() - t0) * 1000

print(f"Computed {len(clusters):,} res6 clusters in {cluster_ms:.0f} ms")
print(f"\nTop 10 clusters by sample count:")
print(clusters[['h3_res6', 'sample_count', 'dominant_source', 'lat', 'lng']].head(10).to_string(index=False))

## Cluster Visualization with Lonboard

Each cluster is rendered as a circle at its hexagon centroid. The radius is proportional
to `log(count)` so that both sparse and dense regions are visible. Colors indicate the
dominant source within each hex cell.

In [ ]:
from lonboard import Map, ScatterplotLayer, BitmapTileLayer

# Build GeoDataFrame from clusters
gdf_clusters = gpd.GeoDataFrame(
    clusters,
    geometry=gpd.points_from_xy(clusters.lng, clusters.lat),
    crs="EPSG:4326"
)

# Colors based on dominant source
colors = np.array([
    SOURCE_COLORS.get(s, DEFAULT_COLOR) for s in gdf_clusters['dominant_source']
], dtype=np.uint8)

# Radius proportional to log(count), scaled for visibility
radii = (np.log1p(gdf_clusters['sample_count'].values) * 800).astype(np.float32)

layer = ScatterplotLayer.from_geopandas(
    gdf_clusters,
    get_fill_color=colors,
    get_radius=radii,
    opacity=0.7,
    radius_min_pixels=2,
)

tile_layer = BitmapTileLayer(
    data="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    min_zoom=0, max_zoom=19,
)

m = Map([tile_layer, layer], view_state={"zoom": 2, "latitude": 20, "longitude": 0})
print(f"Rendering {len(gdf_clusters):,} hex clusters (from {clusters['sample_count'].sum():,} samples)")
print(f"Color legend: Blue=SESAR, Red=OpenContext, Green=GEOME, Orange=Smithsonian")
m

## Multi-Resolution Comparison

Compare clustering at res4 (regional), res6 (city), and res8 (neighborhood) to understand
how granularity affects cluster count and aggregation behavior.

In [ ]:
resolutions = []

for res, col in [(4, 'h3_res4'), (6, 'h3_res6'), (8, 'h3_res8')]:
    t0 = time.time()
    res_stats = con.sql(f"""
        SELECT
            {res} AS resolution,
            COUNT(DISTINCT {col}) AS num_clusters,
            COUNT(*) AS total_points,
            ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT {col}), 1) AS avg_points_per_cluster,
            MAX(ct) AS max_cluster_size,
            MEDIAN(ct) AS median_cluster_size
        FROM (
            SELECT {col}, COUNT(*) AS ct
            FROM read_parquet('{WIDE_H3_URL}')
            WHERE {col} IS NOT NULL
            GROUP BY {col}
        )
    """).df()
    elapsed = (time.time() - t0) * 1000
    res_stats['query_ms'] = round(elapsed)
    resolutions.append(res_stats)

comparison = pd.concat(resolutions, ignore_index=True)
print("Multi-Resolution Comparison")
print("=" * 70)
print(comparison.to_string(index=False))

print("\nGuidance:")
print("  res4 — Best for global/continental overviews (few clusters, fast)")
print("  res6 — Good balance for city-level exploration (~3.2 km hexagons)")
print("  res8 — Detailed neighborhood view (~460 m hexagons, more clusters)")

## Performance: Clusters vs Full Points

Rendering all ~12M points individually overwhelms both the query engine and the browser.
Clustering via H3 reduces data volume by orders of magnitude while preserving spatial patterns.

In [ ]:
# Benchmark: clustered query vs full point query

# Full points (limited to count to avoid browser crash)
t0 = time.time()
full_count = con.sql(f"""
    SELECT COUNT(*) AS n
    FROM read_parquet('{WIDE_H3_URL}')
    WHERE latitude IS NOT NULL
""").fetchone()[0]
full_ms = (time.time() - t0) * 1000

# Clustered at res6
t0 = time.time()
cluster_count = con.sql(f"""
    SELECT COUNT(*) AS n FROM (
        SELECT h3_res6, COUNT(*) AS ct,
               AVG(latitude) AS lat, AVG(longitude) AS lng
        FROM read_parquet('{WIDE_H3_URL}')
        WHERE h3_res6 IS NOT NULL
        GROUP BY h3_res6
    )
""").fetchone()[0]
cluster_ms = (time.time() - t0) * 1000

reduction = full_count / cluster_count if cluster_count > 0 else 0

print("Performance Comparison")
print("=" * 50)
print(f"Full points:     {full_count:>12,} rows  ({full_ms:>6.0f} ms)")
print(f"Res6 clusters:   {cluster_count:>12,} rows  ({cluster_ms:>6.0f} ms)")
print(f"Data reduction:  {reduction:>11.0f}x fewer rows to render")
print(f"\nClustering at res6 reduces rendering payload from ~{full_count/1e6:.1f}M to")
print(f"~{cluster_count/1e3:.0f}K points — enabling smooth interactive maps.")

## Hierarchical Drill-Down: res4 -> res6 -> res8

H3's hierarchy means every res4 cell contains ~49 res6 children, and each res6 cell
contains ~49 res8 children. This enables progressive drill-down — start with a coarse
view, then zoom into regions of interest at finer resolution.

In [ ]:
# Pick the largest res4 cell and drill down
top_res4 = con.sql(f"""
    SELECT h3_res4, COUNT(*) AS cnt
    FROM read_parquet('{WIDE_H3_URL}')
    WHERE h3_res4 IS NOT NULL
    GROUP BY h3_res4
    ORDER BY cnt DESC
    LIMIT 1
""").fetchone()

parent_cell = top_res4[0]
parent_count = top_res4[1]
print(f"Largest res4 cell: {parent_cell} ({parent_count:,} samples)")

# Drill into res6 children
res6_children = con.sql(f"""
    SELECT h3_res6, COUNT(*) AS cnt, MODE(n) AS dominant_source,
           AVG(latitude) AS lat, AVG(longitude) AS lng
    FROM read_parquet('{WIDE_H3_URL}')
    WHERE h3_res4 = {parent_cell}
      AND h3_res6 IS NOT NULL
    GROUP BY h3_res6
    ORDER BY cnt DESC
""").df()

print(f"\nRes4 -> Res6: {len(res6_children)} child cells")
print(res6_children.head(10).to_string(index=False))

# Pick the top res6 child and drill into res8
if len(res6_children) > 0:
    top_res6 = res6_children.iloc[0]
    res8_children = con.sql(f"""
        SELECT h3_res8, COUNT(*) AS cnt, MODE(n) AS dominant_source,
               AVG(latitude) AS lat, AVG(longitude) AS lng
        FROM read_parquet('{WIDE_H3_URL}')
        WHERE h3_res6 = {int(top_res6['h3_res6'])}
          AND h3_res8 IS NOT NULL
        GROUP BY h3_res8
        ORDER BY cnt DESC
    """).df()

    print(f"\nRes6 -> Res8: {len(res8_children)} child cells")
    print(f"(from res6 cell {int(top_res6['h3_res6'])} with {int(top_res6['cnt']):,} samples)")
    print(res8_children.head(10).to_string(index=False))

    print(f"\nDrill-down summary:")
    print(f"  res4: 1 cell -> {parent_count:,} samples")
    print(f"  res6: {len(res6_children)} cells -> {res6_children['cnt'].sum():,} samples")
    print(f"  res8: {len(res8_children)} cells -> {res8_children['cnt'].sum():,} samples (from top res6)")